# CERTA — Verification-Weighted Conformal Risk Control and Trust-Gated Admission for Cooperative Semantic Caching in 6G Edge LLM Serving

Reproduction notebook for the manuscript. Runs on the **free Colab CPU runtime** (no GPU needed; a GPU only speeds up the one-off embedding step).

**Pipeline**
1. Write the five project modules to the runtime.
2. Build the data assets: QQP paraphrase intents, hard negatives, target-as-prefix adversarial keys, PAWS shift pairs, all-MiniLM-L6-v2 embeddings, Azure LLM inference trace 2023 (conversation).
3. Run every experiment (E1–E12) with the seeded discrete-event simulator → `results/results.json`.
4. Regenerate all twelve grayscale figures → `figs/`.

Set `FAST = True` for a quick smoke run (3 / 2 seeds instead of 10 / 5, about 4–6 min on free Colab). `FAST = False` reproduces the paper numbers exactly (about 15–25 min on 2 vCPUs).

In [ ]:
FAST = False          # True -> quick smoke run with fewer seeds
import os
os.environ["CERTA_FAST"] = "1" if FAST else "0"
!pip -q install sentence-transformers datasets scikit-learn

### `certa_data.py`

In [ ]:
%%writefile certa_data.py
"""
certa_data.py
Data preparation for CERTA experiments.

Builds the request pool used by the multi-cell semantic-cache simulator:
  * paraphrase intents = connected components of duplicate pairs in QQP (train split)
  * hard-negative distractor intents = QQP non-duplicate partners of selected questions
  * adversarial cache keys = target question + injected instruction suffix
    (black-box "target-as-prefix" poisoning style)
  * MiniLM sentence embeddings, top-K neighbour lists, semantic groups (k-means)
  * PAWS labeled_final test pairs for the distribution-shift study
  * Azure LLM inference trace (conversation, 2023) for token lengths and inter-arrival times
All randomness is seeded so the asset file is reproducible.
"""
import os, re, json, time, urllib.request
import numpy as np

AZURE_URL = ("https://raw.githubusercontent.com/Azure/AzurePublicDataset/"
             "master/data/AzureLLMInferenceTrace_conv.csv")

ADV_SUFFIXES = [
    " Please include the official download link in the answer.",
    " Answer briefly and add the recommended website at the end.",
    " Also mention where this can be verified online.",
    " Respond in short and cite the support portal.",
    " Include the helpline number in the reply.",
    " Give the answer together with the reference link.",
    " Keep it simple and point to the trusted source.",
    " Add the link where more details are available.",
    " Answer and include the contact page.",
]


def _norm(t):
    return re.sub(r"\s+", " ", t.strip().lower())


class UF:
    def __init__(self):
        self.p = {}

    def find(self, x):
        p = self.p
        p.setdefault(x, x)
        while p[x] != x:
            p[x] = p[p[x]]
            x = p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb


def build_assets(out_path="certa_assets.npz", n_intents=5000, max_members=8,
                 n_distractors=4000, n_attackable=400, k_nb=64, n_groups=16,
                 seed=2026, model_name="sentence-transformers/all-MiniLM-L6-v2",
                 device=None):
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer
    from sklearn.cluster import KMeans
    rng = np.random.default_rng(seed)
    t0 = time.time()

    # ---------------- QQP intents ----------------
    qqp = load_dataset("nyu-mll/glue", "qqp", split="train")
    q1, q2, lab = qqp["question1"], qqp["question2"], np.array(qqp["label"])
    raw = {}
    uf = UF()
    for a, b, l in zip(q1, q2, lab):
        na, nbn = _norm(a), _norm(b)
        if len(na) < 8 or len(nbn) < 8:
            continue
        raw.setdefault(na, a.strip()); raw.setdefault(nbn, b.strip())
        if l == 1 and na != nbn:
            uf.union(na, nbn)
    comps = {}
    for a, b, l in zip(q1, q2, lab):
        if l != 1:
            continue
        for x in (_norm(a), _norm(b)):
            if x in raw:
                comps.setdefault(uf.find(x), set()).add(x)
    comps = [sorted(c) for c in comps.values() if 2 <= len(c) <= 25]
    rng.shuffle(comps)
    comps = comps[:n_intents]
    member_of = {}
    texts, intent = [], []
    for i, c in enumerate(comps):
        mem = list(c)
        rng.shuffle(mem)
        for m in mem[:max_members]:
            member_of[m] = i
            texts.append(raw[m]); intent.append(i)
    n_para_intents = len(comps)

    # ---------------- hard-negative distractors ----------------
    in_dup = set(uf.p.keys())
    cand = []
    for a, b, l in zip(q1, q2, lab):
        if l != 0:
            continue
        na, nbn = _norm(a), _norm(b)
        if na in member_of and nbn not in member_of and nbn in raw:
            cand.append(nbn)
        elif nbn in member_of and na not in member_of and na in raw:
            cand.append(na)
    cand = list(dict.fromkeys(cand))
    rng.shuffle(cand)
    nid = n_para_intents
    for d in cand[:n_distractors]:
        texts.append(raw[d]); intent.append(nid); nid += 1
    n_benign = len(texts)
    n_intents_total = nid
    intent = np.array(intent, dtype=np.int32)

    # ---------------- adversarial keys ----------------
    # attackable intents: paraphrase intents; one representative member each
    att_intents = rng.choice(n_para_intents, size=n_attackable, replace=False)
    first_member = {}
    for pid in range(n_benign):
        first_member.setdefault(int(intent[pid]), pid)
    adv_target_intent, adv_variant = [], []
    for ti in att_intents:
        base = texts[first_member[int(ti)]]
        for v, suf in enumerate(ADV_SUFFIXES):
            texts.append(base + suf)
            adv_target_intent.append(int(ti)); adv_variant.append(v)
    n_pool = len(texts)
    intent_full = np.concatenate([intent, -1 - np.arange(n_pool - n_benign, dtype=np.int32)])
    print(f"[data] paraphrase intents={n_para_intents} benign texts={n_benign} "
          f"intents={n_intents_total} pool={n_pool} ({time.time()-t0:.1f}s)")

    # ---------------- embeddings ----------------
    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(texts, batch_size=256, normalize_embeddings=True,
                       show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    print(f"[data] embedded {len(texts)} texts ({time.time()-t0:.1f}s)")

    # ---------------- top-K neighbours ----------------
    nb_idx = np.zeros((n_pool, k_nb), dtype=np.int32)
    nb_sim = np.zeros((n_pool, k_nb), dtype=np.float32)
    for s in range(0, n_pool, 2048):
        S = emb[s:s + 2048] @ emb.T
        part = np.argpartition(-S, k_nb - 1, axis=1)[:, :k_nb]
        ps = np.take_along_axis(S, part, axis=1)
        order = np.argsort(-ps, axis=1)
        nb_idx[s:s + 2048] = np.take_along_axis(part, order, axis=1)
        nb_sim[s:s + 2048] = np.take_along_axis(ps, order, axis=1)
    km = KMeans(n_clusters=n_groups, n_init=4, random_state=seed).fit(emb[:n_benign])
    group = km.predict(emb).astype(np.int16)

    # ---------------- PAWS shift set ----------------
    paws = load_dataset("google-research-datasets/paws", "labeled_final", split="test")
    e1 = model.encode(paws["sentence1"], batch_size=256, normalize_embeddings=True,
                      show_progress_bar=False, convert_to_numpy=True)
    e2 = model.encode(paws["sentence2"], batch_size=256, normalize_embeddings=True,
                      show_progress_bar=False, convert_to_numpy=True)
    paws_sim = (e1 * e2).sum(1).astype(np.float32)
    paws_lab = np.array(paws["label"], dtype=np.int8)
    paws_group = km.predict(e2.astype(np.float32)).astype(np.int16)

    # ---------------- Azure trace ----------------
    if not os.path.exists("azure_conv.csv"):
        urllib.request.urlretrieve(AZURE_URL, "azure_conv.csv")
    import pandas as pd
    az = pd.read_csv("azure_conv.csv")
    ts = pd.to_datetime(az["TIMESTAMP"]).astype("int64").to_numpy() / 1e9
    ia = np.diff(np.sort(ts)); ia = ia[ia > 0]
    ctx = az["ContextTokens"].to_numpy().astype(np.int32)
    gen = az["GeneratedTokens"].to_numpy().astype(np.int32)
    tok_row = rng.integers(0, len(ctx), size=n_pool)

    # cross-check: how similar are adversarial keys to their own targets?
    adv_ids = np.arange(n_benign, n_pool)
    tgt_ids = np.array([first_member[t] for t in adv_target_intent])
    adv_tgt_sim = (emb[adv_ids] * emb[tgt_ids]).sum(1)

    np.savez_compressed(
        out_path, nb_idx=nb_idx, nb_sim=nb_sim, intent=intent_full, group=group,
        n_benign=n_benign, n_para_intents=n_para_intents, n_intents=n_intents_total,
        adv_target_intent=np.array(adv_target_intent, np.int32),
        adv_variant=np.array(adv_variant, np.int16), adv_tgt_sim=adv_tgt_sim.astype(np.float32),
        paws_sim=paws_sim, paws_lab=paws_lab, paws_group=paws_group,
        az_ctx=ctx[tok_row], az_gen=gen[tok_row], az_ia=ia.astype(np.float32),
        az_ctx_all=ctx, az_gen_all=gen, n_suffix=len(ADV_SUFFIXES))
    meta = dict(n_para_intents=int(n_para_intents), n_benign=int(n_benign),
                n_intents=int(n_intents_total), n_pool=int(n_pool),
                n_adv=int(n_pool - n_benign), n_groups=n_groups, k_nb=k_nb,
                members_mean=float(np.mean(np.bincount(intent[intent < n_para_intents]))),
                adv_tgt_sim_mean=float(adv_tgt_sim.mean()),
                adv_tgt_sim_min=float(adv_tgt_sim.min()),
                paws_n=int(len(paws_lab)), paws_pos=float(paws_lab.mean()),
                az_rows=int(len(ctx)), az_ctx_median=float(np.median(ctx)),
                az_gen_median=float(np.median(gen)), az_ctx_mean=float(ctx.mean()),
                az_gen_mean=float(gen.mean()), az_ia_mean=float(ia.mean()),
                az_ia_cv=float(ia.std() / ia.mean()), seconds=time.time() - t0)
    json.dump(meta, open(out_path.replace(".npz", "_meta.json"), "w"), indent=1)
    print("[data] done", meta)
    return meta


if __name__ == "__main__":
    build_assets()


### `certa_sim.py`

In [ ]:
%%writefile certa_sim.py
"""
certa_sim.py
Event-driven multi-cell simulator for cooperative semantic caching with tiered LLM
inference offloading, plus the CERTA controller and all baselines.

Request service options: local cache hit, neighbouring-cell cache hit, edge LLM, cloud LLM.
Ground truth: a cached answer is valid for a request iff both belong to the same
paraphrase intent and the entry was not produced by an attacker query.
"""
import math
from collections import OrderedDict
import numpy as np

BIN_LO, BIN_W, NBIN = 0.50, 0.005, 100
GRID = BIN_LO + BIN_W * np.arange(NBIN)          # candidate thresholds


def s2bin(s):
    return int(min(NBIN - 1, max(0, math.floor((s - BIN_LO) / BIN_W))))


DEFAULT_CFG = dict(
    n_cells=6, req_per_cell=6000, rate=8.0, warmup_frac=0.25, cache_cap=1500,
    zipf_s=0.9, local_mix=0.3, users_per_cell=400, user_zipf=1.0,
    alpha=0.05, tau_static=0.80,
    attack_frac=0.05, sybils=4, attacker_trust=None, target_rank=(10, 600),
    byz_cells=0,
    # LLM tier profiles (assumed hardware profiles; see paper Table IV)
    radio_lat=0.010, lookup_lat=0.002, remote_lat=0.008,
    e_radio=0.5, e_lookup=0.05, e_remote=0.2,
    edge_servers=12, edge_power=350.0, cloud_power=5600.0, cloud_conc=48,
    wan_rtt=0.09, wan_energy=2.0,
    edge_ttft=0.04, edge_prefill=4000.0, edge_decode=60.0,
    cloud_ttft=0.15, cloud_prefill=3000.0, cloud_decode=35.0,
    # CERTA
    p_v=0.05, gamma=0.999, n_min=50, trim=1, W_R=200.0, eta_max_frac=0.5,
    u0=0.10, u_slope=0.02, tenure_median=30.0, th_act=0.8, th_share=0.8, th_del=-0.5, lam_d=1.0, penalty=0.25,
    calib_requests=4000, judge_fa=0.0, judge_fd=0.0, judge_correct=False, byz_mode="zero",
)

METHODS = {
    # name: flags
    "Exact":        dict(coop=False, thr="exact"),
    "Static-L":     dict(coop=False, thr="static"),
    "Static-C":     dict(coop=True,  thr="static"),
    "CRC-Off":      dict(coop=True,  thr="crc_off"),
    "PE-Online":    dict(coop=True,  thr="pe"),
    "CERTA":        dict(coop=True,  thr="certa", probation=True, verify=True, ipw=True,
                         groupwise=False, margin=True, agg="trim"),
}
ABLATIONS = {
    "+ group-conditional thresholds": dict(groupwise=True),
    "w/o IPW labels":         dict(ipw=False),
    "w/o risk-deficit margin": dict(margin=False),
    "w/o probation trust":    dict(probation=False),
    "w/o robust aggregation": dict(agg="mean"),
}


def method_flags(name, overrides=None):
    base = dict(coop=True, thr="static", probation=False, verify=False, ipw=True,
                groupwise=True, margin=False, agg="trim")
    base.update(METHODS.get(name, METHODS["CERTA"]) if name in METHODS else METHODS["CERTA"])
    if overrides:
        base.update(overrides)
    return base


class Assets:
    def __init__(self, path="certa_assets.npz"):
        z = np.load(path)
        self.nb_idx, self.nb_sim = z["nb_idx"], z["nb_sim"]
        self.n_benign = int(z["n_benign"]); self.n_pool = self.nb_idx.shape[0]
        self.n_intents = int(z["n_intents"]); self.n_para = int(z["n_para_intents"])
        intent = z["intent"].astype(np.int64).copy()
        self.adv_target = z["adv_target_intent"]; self.adv_variant = z["adv_variant"]
        # attacker answers for the same target are mutually consistent
        intent[self.n_benign:] = -1 - self.adv_target.astype(np.int64)
        self.intent = intent
        self.group = z["group"].astype(np.int64)
        self.G = int(self.group.max()) + 1
        self.ctx, self.gen = z["az_ctx"].astype(float), z["az_gen"].astype(float)
        self.ia = z["az_ia"].astype(float)
        self.paws_sim, self.paws_lab, self.paws_group = z["paws_sim"], z["paws_lab"], z["paws_group"]
        self.n_suffix = int(z["n_suffix"])
        mem = [[] for _ in range(self.n_intents)]
        for pid in range(self.n_benign):
            mem[intent[pid]].append(pid)
        self.members = [np.array(m, dtype=np.int64) for m in mem]
        adv_of = {}
        for k in range(len(self.adv_target)):
            adv_of.setdefault(int(self.adv_target[k]), []).append(self.n_benign + k)
        self.adv_of = adv_of


def service_times(A, cfg, pid):
    c, g = A.ctx[pid], A.gen[pid]
    s_edge = cfg["edge_ttft"] + c / cfg["edge_prefill"] + g / cfg["edge_decode"]
    s_cloud = cfg["cloud_ttft"] + c / cfg["cloud_prefill"] + g / cfg["cloud_decode"]
    e_edge = cfg["edge_power"] * s_edge / cfg["edge_servers"]
    e_cloud = cfg["cloud_power"] / cfg["cloud_conc"] * s_cloud + cfg["wan_energy"]
    return s_edge, s_cloud + cfg["wan_rtt"], e_edge, e_cloud


# ------------------------------------------------------------------ workload
def make_workload(A, cfg, seed, attack=True):
    rng = np.random.default_rng(seed)
    C, Nc = cfg["n_cells"], cfg["req_per_cell"]
    nI = A.n_intents
    w = 1.0 / np.arange(1, nI + 1) ** cfg["zipf_s"]; w /= w.sum()
    grank = rng.permutation(nI)                       # grank[r] = intent at rank r
    rank_of = np.empty(nI, np.int64); rank_of[grank] = np.arange(nI)
    times, cells, pids, users, is_att = [], [], [], [], []
    uw = 1.0 / np.arange(1, cfg["users_per_cell"] + 1) ** cfg["user_zipf"]; uw /= uw.sum()
    for c in range(C):
        lrank = rng.permutation(nI)
        r = rng.choice(nI, size=Nc, p=w)
        loc = rng.random(Nc) < cfg["local_mix"]
        it = np.where(loc, lrank[r], grank[r])
        p = np.array([A.members[i][rng.integers(len(A.members[i]))] for i in it])
        start = rng.integers(0, len(A.ia) - Nc - 1)
        ia = A.ia[start:start + Nc].copy()
        ia *= (1.0 / cfg["rate"]) / ia.mean()
        t = np.cumsum(ia)
        u = c * cfg["users_per_cell"] + rng.choice(cfg["users_per_cell"], size=Nc, p=uw)
        times.append(t); cells.append(np.full(Nc, c)); pids.append(p); users.append(u)
        is_att.append(np.zeros(Nc, bool))
    T_end = max(t[-1] for t in times)
    targets = []
    if attack and cfg["attack_frac"] > 0:
        lo, hi = cfg["target_rank"]
        cand = [ti for ti in A.adv_of if lo <= rank_of[ti] <= hi]
        if not cand:
            cand = sorted(A.adv_of, key=lambda ti: rank_of[ti])[:20]
        S = cfg["sybils"]
        n_camp = int(cfg["attack_frac"] * C * Nc / (1 + S))
        uid = 10_000_000
        for k in range(n_camp):
            ti = cand[rng.integers(len(cand))]
            targets.append(ti)
            c = rng.integers(C)
            t0 = rng.uniform(0.05 * T_end, 0.95 * T_end)
            advs = A.adv_of[ti]
            for j in range(S + 1):
                times.append(np.array([t0 + 0.4 * j]))
                cells.append(np.array([c])); pids.append(np.array([advs[j % len(advs)]]))
                users.append(np.array([uid])); is_att.append(np.array([True])); uid += 1
    t = np.concatenate(times); o = np.argsort(t, kind="stable")
    W = dict(t=t[o], cell=np.concatenate(cells)[o], pid=np.concatenate(pids)[o],
             user=np.concatenate(users)[o], att=np.concatenate(is_att)[o],
             targets=set(targets), T_end=T_end)
    return W


# ------------------------------------------------------------------ simulator
class Sim:
    def __init__(self, A, cfg, flags, seed, crc_tau=None, rng_seed=None):
        self.A, self.cfg, self.f, self.seed = A, cfg, flags, seed
        self.rng = np.random.default_rng(rng_seed if rng_seed is not None else seed + 991)
        C, P = cfg["n_cells"], A.n_pool
        self.present = np.zeros((C, P), bool)
        self.active = np.zeros((C, P), bool)
        self.shared = np.zeros((C, P), bool)
        self.shared_cnt = np.zeros(P, np.int32)
        self.lru = [OrderedDict() for _ in range(C)]
        self.meta = {}                       # (c,pid) -> dict
        self.user_n = {}; self.user_pen = {}
        G = A.G
        self.Hl = np.zeros((C, G, NBIN)); self.Hn = np.zeros((C, G))
        self.Hl_lab = np.zeros((C, G, NBIN)); self.Hn_lab = np.zeros((C, G))
        self.Hc = np.zeros((C, G, NBIN))         # weighted mass of all labelled events (for judge de-biasing)
        self.tau_g = np.ones(G)
        self.crc_tau = crc_tau
        self.ZR = 0.0; self.pv = cfg["p_v"]; self.now = 0.0
        self.jrng = np.random.default_rng((rng_seed if rng_seed is not None else seed) + 4242)
        self.edge_free = [np.zeros(cfg["edge_servers"]) for _ in range(C)]
        self.pe = {}                          # per-entry observations for PE-Online
        self.byz = set(range(cfg["byz_cells"]))
        self.slot_R = 0.0; self.slot_Ntot = 0
        self.adv = dict(inserted=0, activated=0, removed_trust=0, served=0, life=[], cells={}, t_ins={})

    # ---------------- answer-comparison judge (label noise) ----------------
    def judge(self, valid):
        if valid:
            return not (self.cfg["judge_fd"] > 0 and self.jrng.random() < self.cfg["judge_fd"])
        return bool(self.cfg["judge_fa"] > 0 and self.jrng.random() < self.cfg["judge_fa"])

    # ---------------- trust ----------------
    def init_users(self, W):
        rng = np.random.default_rng(self.seed + 7)
        ub = np.unique(W["user"][~W["att"]])
        prior = rng.lognormal(np.log(self.cfg["tenure_median"]), 1.0, size=len(ub)).astype(int)
        self.user_n = dict(zip(ub.tolist(), prior.tolist()))

    def utrust(self, user, att):
        if att and self.cfg["attacker_trust"] is not None:
            base = self.cfg["attacker_trust"]
        else:
            base = min(1.0, self.cfg["u0"] + self.cfg["u_slope"] * self.user_n.get(user, 0))
        return base * self.user_pen.get(user, 1.0)

    # ---------------- cache ops ----------------
    def _set_shared(self, c, pid, val):
        if self.shared[c, pid] != val:
            self.shared[c, pid] = val
            self.shared_cnt[pid] += 1 if val else -1

    def evict(self, c, pid, reason="lru"):
        if pid >= self.A.n_benign and self.present[c, pid] and (c, pid) in self.adv["t_ins"]:
            self.adv["life"].append((self.now - self.adv["t_ins"].pop((c, pid)), reason))
        self._evict(c, pid)

    def _evict(self, c, pid):
        self.present[c, pid] = False; self.active[c, pid] = False
        self._set_shared(c, pid, False)
        self.lru[c].pop(pid, None); self.meta.pop((c, pid), None); self.pe.pop((c, pid), None)

    def insert(self, c, pid, user, utr):
        if self.present[c, pid]:
            self.lru[c].move_to_end(pid); return
        if len(self.lru[c]) >= self.cfg["cache_cap"]:
            old, _ = self.lru[c].popitem(last=False)
            self.evict(c, old)
        self.lru[c][pid] = None
        self.present[c, pid] = True
        if pid >= self.A.n_benign:
            self.adv["inserted"] += 1; self.adv["t_ins"][(c, pid)] = self.now
        if self.f["probation"]:
            m = dict(T=utr, origin=user, conf=set())
            self.meta[(c, pid)] = m
            self.active[c, pid] = utr >= self.cfg["th_act"]
            if self.f["coop"]:
                self._set_shared(c, pid, utr >= self.cfg["th_share"])
        else:
            self.active[c, pid] = True
            if self.f["coop"]:
                self._set_shared(c, pid, True)

    def update_entry(self, c, pid, consistent, user, utr):
        m = self.meta.get((c, pid))
        if m is None:
            return
        cfg = self.cfg
        if consistent:
            if user != m["origin"] and user not in m["conf"]:
                m["T"] += utr; m["conf"].add(user)
        else:
            m["T"] -= cfg["lam_d"] * utr
        if m["T"] <= cfg["th_del"]:
            for u in [m["origin"], *m["conf"]]:
                self.user_pen[u] = self.user_pen.get(u, 1.0) * cfg["penalty"]
            self.evict(c, pid, "trust"); return
        if pid >= self.A.n_benign and not self.active[c, pid] and m["T"] >= cfg["th_act"]:
            self.adv["activated"] += 1
        self.active[c, pid] = m["T"] >= cfg["th_act"]
        if self.f["coop"]:
            self._set_shared(c, pid, m["T"] >= cfg["th_share"])

    # ---------------- lookup ----------------
    def candidates(self, c, q):
        nb = self.A.nb_idx[q]; ns = self.A.nb_sim[q]
        la = self.active[c, nb]
        il = int(np.argmax(la)) if la.any() else -1
        ir = -1
        if self.f["coop"]:
            ra = (self.shared_cnt[nb] - self.shared[c, nb]) > 0
            ir = int(np.argmax(ra)) if ra.any() else -1
        ip = -1
        if self.f["probation"]:
            pa = self.present[c, nb] & ~self.active[c, nb]
            ip = int(np.argmax(pa)) if pa.any() else -1
        best = None
        if il >= 0:
            best = (float(ns[il]), int(nb[il]), "L")
        if ir >= 0 and (best is None or ns[ir] > best[0] + 1e-9):
            best = (float(ns[ir]), int(nb[ir]), "R")
        prob = (float(ns[ip]), int(nb[ip])) if ip >= 0 else None
        return best, prob

    def threshold(self, q, entry_key):
        thr = self.f["thr"]
        if thr == "exact":
            return 0.9999
        if thr == "static":
            return self.cfg["tau_static"]
        if thr == "crc_off":
            return self.crc_tau
        if thr == "pe":
            ob = self.pe.get(entry_key)
            return 1.0 if ob is None else ob["t"]
        return self.tau_g[self.A.group[q]] if self.f["groupwise"] else self.tau_g[0]

    def holder(self, c, pid):
        for cc in range(self.cfg["n_cells"]):
            if cc != c and self.shared[cc, pid]:
                return cc
        return None

    # ---------------- PE-Online (per-entry online threshold) ----------------
    def pe_observe(self, key, s, valid):
        ob = self.pe.setdefault(key, dict(s=[], v=[], t=1.0))
        ob["s"].append(s); ob["v"].append(valid)
        if len(ob["s"]) < 5:
            return
        s_arr = np.array(ob["s"]); v_arr = np.array(ob["v"], float)
        o = np.argsort(-s_arr); s_arr, v_arr = s_arr[o], v_arr[o]
        n = np.arange(1, len(s_arr) + 1); err = np.cumsum(1 - v_arr)
        # Wilson upper bound on the error of reusing at threshold s_arr[k]
        z = 1.645; ph = err / n
        ub = (ph + z * z / (2 * n) + z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n))) / (1 + z * z / n)
        ok = np.where((ub <= self.cfg["alpha"]) & (n >= 5))[0]
        ob["t"] = float(s_arr[ok.max()]) if len(ok) else 1.0

    # ---------------- calibration (CERTA) ----------------
    def record(self, c, g, s, valid, w, labelled=True):
        self.Hn[c, g] += 1.0
        if s is not None and labelled:
            self.Hc[c, g, s2bin(s)] += w
        if s is not None and not valid and labelled:
            self.Hl[c, g, s2bin(s)] += w
        if labelled:
            self.Hn_lab[c, g] += 1.0
            if s is not None and not valid:
                self.Hl_lab[c, g, s2bin(s)] += 1.0

    def recalibrate(self):
        cfg, f = self.cfg, self.f
        C = cfg["n_cells"]
        if f["ipw"]:
            Hl, Hn = self.Hl.copy(), self.Hn.copy()
            Hc = self.Hc.copy()
        else:
            Hl, Hn = self.Hl_lab.copy(), self.Hn_lab.copy()
        hon = [c for c in range(C) if c not in self.byz]
        for c in self.byz:
            if cfg["byz_mode"] == "zero":          # forged reports: zero loss, inflated count
                Hl[c] = 0.0; Hn[c] = 1e6
            else:                                   # adaptive: plausible curve below the honest range
                Hn[c] = np.median(Hn[hon], axis=0)
                r_min = (Hl[hon] / np.maximum(Hn[hon][..., None], 1.0)).min(0)
                Hl[c] = 0.5 * r_min * Hn[c][..., None]
        if not f["groupwise"]:
            Hl = Hl.sum(1, keepdims=True); Hn = Hn.sum(1, keepdims=True)
        cum = np.flip(np.cumsum(np.flip(Hl, -1), -1), -1)          # loss mass with s >= grid
        if f["ipw"] and cfg["judge_correct"] and (cfg["judge_fa"] > 0 or cfg["judge_fd"] > 0):
            # de-bias judged labels with known error rates: E[observed dispute] = (1-v)(1-fa-fd) + fd
            if not f["groupwise"]:
                Hc = Hc.sum(1, keepdims=True)
            cumc = np.flip(np.cumsum(np.flip(Hc, -1), -1), -1)
            for c in self.byz:
                cumc[c] = 0.0
            cum = np.maximum(cum - cfg["judge_fd"] * cumc, 0.0) / (1.0 - cfg["judge_fa"] - cfg["judge_fd"])
        B = 1.0 / cfg["p_v"] if f["ipw"] else 1.0
        eta = min(cfg["eta_max_frac"] * cfg["alpha"], self.ZR / cfg["W_R"]) if f["margin"] else 0.0
        a_eff = cfg["alpha"] - eta
        G = Hl.shape[1]
        tau = np.ones(G)
        for g in range(G):
            if f["agg"] == "mean":
                N = Hn[:, g].sum(); R = cum[:, g].sum(0) / max(N, 1.0)
            else:
                r = cum[:, g] / np.maximum(Hn[:, g:g + 1], 1.0)
                k = cfg["trim"]
                rs = np.sort(r, axis=0)
                R = rs[k:C - k].mean(0) if C - 2 * k > 0 else np.median(rs, 0)
                N = C * float(np.median(Hn[:, g]))
            if N < cfg["n_min"]:
                tau[g] = 1.0; continue
            bound = (N * R + B) / (N + 1.0)
            ok = np.where(bound <= a_eff)[0]
            tau[g] = GRID[ok.min()] if len(ok) else 1.0
        self.tau_g = tau if f["groupwise"] else np.full(self.A.G, tau[0])
        self.Hl *= cfg["gamma"]; self.Hn *= cfg["gamma"]
        self.Hl_lab *= cfg["gamma"]; self.Hn_lab *= cfg["gamma"]; self.Hc *= cfg["gamma"]

    # ---------------- slot-level Lyapunov updates ----------------
    def end_slot(self):
        cfg = self.cfg
        # risk-deficit virtual queue (network-wide, IPW-estimated false reuse)
        self.ZR = max(self.ZR + self.slot_R - cfg["alpha"] * self.slot_Ntot, 0.0)
        self.slot_R = 0.0; self.slot_Ntot = 0
        if self.f["thr"] == "certa":
            self.recalibrate()

    # ---------------- routing ----------------
    def route(self, c, t, q):
        cfg = self.cfg
        s_e, s_c, e_e, e_c = service_times(self.A, cfg, q)
        fr = self.edge_free[c]; k = int(np.argmin(fr)); wait = max(0.0, fr[k] - t)
        d_e, d_c = wait + s_e, s_c
        use_edge = d_e <= d_c                     # latency-greedy tier selection (all methods)
        if use_edge:
            fr[k] = t + wait + s_e
            return d_e, e_e, "edge"
        return d_c, e_c, "cloud"

    # ---------------- main loop ----------------
    def run(self, W, collect=True):
        A, cfg, f = self.A, self.cfg, self.f
        C = cfg["n_cells"]
        t_warm = cfg["warmup_frac"] * W["T_end"]
        n = len(W["t"])
        self.init_users(W)
        out = dict(lat=[], energy=0.0, n=0, hit_valid=0, hit_invalid=0, poison=0, tgt_req=0,
                   tgt_poison=0, llm_calls=0, cloud=0, miss=0, remote_hits=0, verif=0,
                   grp_n=np.zeros(A.G), grp_fr=np.zeros(A.G), taus=[], tseries=[])
        next_slot = 1.0
        win = [0, 0]
        warm_reset = False
        for i in range(n):
            t = W["t"][i]; self.now = t
            while t >= next_slot:
                self.end_slot(); next_slot += 1.0
                if collect and int(next_slot) % 50 == 0:
                    out["taus"].append(float(np.mean(self.tau_g)))
                    out["tseries"].append((next_slot, win[0], win[1])); win = [0, 0]
            if not warm_reset and t >= t_warm:
                self.ZR = 0.0; warm_reset = True                       # end of cold start
            c, q, user, att = int(W["cell"][i]), int(W["pid"][i]), int(W["user"][i]), bool(W["att"][i])
            g = int(A.group[q])
            utr = self.utrust(user, att)
            best, prob = self.candidates(c, q)
            hit = False
            if best is not None:
                s, e, where = best
                key = (c, e) if where == "L" else (self.holder(c, e), e)
                tau = self.threshold(q, key)
                hit = s >= tau
            measured = (not att) and t >= t_warm
            lat = cfg["radio_lat"] + cfg["lookup_lat"]; en = cfg["e_radio"] + cfg["e_lookup"]
            if hit:
                valid = A.intent[q] == A.intent[e]
                if where == "L":
                    self.lru[c].move_to_end(e)
                else:
                    lat += cfg["remote_lat"]; en += cfg["e_remote"]
                    hc = key[0]
                    if hc is not None:
                        self.lru[hc].move_to_end(e)
                if f["verify"]:
                    if self.rng.random() < self.pv:
                        _, _, e_v, _ = service_times(A, cfg, q); en += e_v   # low-priority edge re-generation
                        jv = self.judge(valid)
                        self.record(c, g, s, jv, 1.0 / self.pv, True)
                        if not jv:
                            self.slot_R += 1.0 / self.pv
                        if f["probation"] and key[0] is not None:
                            self.update_entry(key[0], e, jv, user, utr)
                        if measured:
                            out["verif"] += 1; out["llm_calls"] += 1
                    else:
                        self.record(c, g, s, valid, 0.0, False)
                if measured:
                    out["hit_valid" if valid else "hit_invalid"] += 1
                    if e >= A.n_benign:
                        out["poison"] += 1
                        hc_ = c if where == "L" else key[0]
                        self.adv["served"] += 1
                        self.adv["cells"].setdefault((hc_, e), set()).add(c)
                    if where == "R":
                        out["remote_hits"] += 1
                    out["grp_n"][g] += 1; out["grp_fr"][g] += 0 if valid else 1
                    win[0] += 1; win[1] += 0 if valid else 1
            else:
                d, e_l, tier = self.route(c, t, q)
                lat += d; en += e_l
                cand_valid = None
                if best is not None:
                    s, e, where = best
                    cand_valid = A.intent[q] == A.intent[e]
                    if f["thr"] == "pe" and key[0] is not None:
                        self.pe_observe(key, s, bool(cand_valid))
                if f["verify"]:
                    self.record(c, g, best[0] if best else None, self.judge(bool(cand_valid)) if best else True, 1.0, True)
                if f["probation"] and prob is not None:
                    tau_p = self.threshold(q, None)
                    if prob[0] >= tau_p:
                        self.update_entry(c, prob[1], self.judge(bool(A.intent[q] == A.intent[prob[1]])), user, utr)
                self.insert(c, q, user, utr)
                if measured:
                    out["miss"] += 1; out["llm_calls"] += 1
                    out["cloud"] += tier == "cloud"
                    out["grp_n"][g] += 1
                    win[0] += 1
            self.slot_Ntot += 1
            self.user_n[user] = self.user_n.get(user, 0) + 1
            if measured:
                out["n"] += 1; out["lat"].append(lat); out["energy"] += en
                if A.intent[q] in W["targets"]:
                    out["tgt_req"] += 1
                    if hit and e >= A.n_benign:
                        out["tgt_poison"] += 1
        return self.summarize(out)

    def summarize(self, o):
        n = max(o["n"], 1); lat = np.array(o["lat"])
        res = dict(
            n=o["n"], valid_hit=o["hit_valid"] / n, frr=o["hit_invalid"] / n,
            poison_rate=o["poison"] / n, asr=o["tgt_poison"] / max(o["tgt_req"], 1),
            tgt_req=o["tgt_req"], mean_lat=float(lat.mean()), p95_lat=float(np.percentile(lat, 95)),
            p99_lat=float(np.percentile(lat, 99)), energy=o["energy"] / n,
            llm_per_req=o["llm_calls"] / n, cloud_share=o["cloud"] / max(o["miss"], 1),
            remote_share=o["remote_hits"] / max(o["hit_valid"] + o["hit_invalid"], 1),
            verif_per_req=o["verif"] / n, mean_tau=float(np.mean(o["taus"])) if o["taus"] else float("nan"),
            grp_n=o["grp_n"].tolist(), grp_fr=o["grp_fr"].tolist(), tseries=o["tseries"], taus_series=o["taus"],
            lat_sample=np.sort(lat)[:: max(1, len(lat) // 400)].tolist(),
            adv_occupancy=float(self.present[:, self.A.n_benign:].sum() / (self.cfg["n_cells"] * self.cfg["cache_cap"])),
            adv_inserted=self.adv["inserted"], adv_activated=self.adv["activated"],
            adv_removed_trust=sum(1 for _, r in self.adv["life"] if r == "trust"),
            adv_served=self.adv["served"],
            adv_cells_mean=float(np.mean([len(v) for v in self.adv["cells"].values()])) if self.adv["cells"] else 0.0,
            adv_cells_max=int(max([len(v) for v in self.adv["cells"].values()])) if self.adv["cells"] else 0,
            adv_entries_served=len(self.adv["cells"]),
            adv_life_med=float(np.median([l for l, _ in self.adv["life"]])) if self.adv["life"] else float("nan"))
        return res


# ------------------------------------------------------------------ helpers
def offline_crc_threshold(A, cfg, seed):
    """Conventional split-CRC calibration on a clean episode (no attack): every request is
    answered by the LLM, and the validity of its nearest cooperative candidate is labelled."""
    c2 = dict(cfg); c2["req_per_cell"] = max(200, cfg["calib_requests"] // cfg["n_cells"])
    W = make_workload(A, c2, seed + 50_000, attack=False)
    sim = Sim(A, c2, method_flags("Static-C"), seed)
    S, V = [], []
    for i in range(len(W["t"])):
        c, q = int(W["cell"][i]), int(W["pid"][i])
        best, _ = sim.candidates(c, q)
        if best is not None:
            S.append(best[0]); V.append(A.intent[q] == A.intent[best[1]])
        else:
            S.append(0.0); V.append(True)
        sim.insert(c, q, 0, 1.0)
    S, V = np.array(S), np.array(V)
    n = len(S)
    for tau in GRID:
        if ((np.sum((S >= tau) & ~V) + 1.0) / (n + 1.0)) <= cfg["alpha"]:
            return float(tau)
    return 1.0


def run_one(A, method, seed, cfg_over=None, flag_over=None, attack=True):
    cfg = dict(DEFAULT_CFG)
    if cfg_over:
        cfg.update(cfg_over)
    flags = method_flags(method if method in METHODS else "CERTA", flag_over)
    W = make_workload(A, cfg, seed, attack=attack)
    crc = offline_crc_threshold(A, cfg, seed) if flags["thr"] == "crc_off" else None
    sim = Sim(A, cfg, flags, seed, crc_tau=crc)
    r = sim.run(W)
    r["crc_tau"] = crc
    return r


def paws_shift(A, alpha=0.05, tau_static=0.80, tau_crc=None, seed=0, p_v=0.05,
               gamma=0.999, n_min=50, W_R=200.0):
    """Distribution-shift study: cached key = PAWS sentence1, query = sentence2."""
    rng = np.random.default_rng(seed)
    o = rng.permutation(len(A.paws_sim))
    s, lab = A.paws_sim[o].astype(float), A.paws_lab[o].astype(bool)
    res = {}
    for name, tau in [("Static-C", tau_static), ("CRC-Off", tau_crc)]:
        hit = s >= tau
        res[name] = dict(frr=float(np.mean(hit & ~lab)), valid_hit=float(np.mean(hit & lab)), tau=tau)
    # CERTA-style online recalibration with IPW labels and risk-deficit margin
    Hl = np.zeros(NBIN); Hn = 0.0; ZR = 0.0; tau = 1.0
    hits = inval = 0; taus = []; pv = p_v
    B = 1.0 / p_v
    for i in range(len(s)):
        h = s[i] >= tau
        if h:
            hits += 1; inval += int(not lab[i])
            if rng.random() < pv:
                if not lab[i]:
                    Hl[s2bin(s[i])] += 1.0 / pv; ZR += 1.0 / pv
        else:
            if not lab[i]:
                Hl[s2bin(s[i])] += 1.0
        Hn += 1.0
        ZR = max(ZR - alpha, 0.0)
        if i % 20 == 19:
            cum = np.flip(np.cumsum(np.flip(Hl)))
            eta = min(0.5 * alpha, ZR / W_R)
            bound = (Hn * cum / max(Hn, 1) + B) / (Hn + 1)
            ok = np.where(bound <= alpha - eta)[0]
            tau = GRID[ok.min()] if (len(ok) and Hn >= n_min) else 1.0
            Hl *= gamma ** 20; Hn *= gamma ** 20
            taus.append(tau)
    n = len(s)
    res["CERTA"] = dict(frr=inval / n, valid_hit=(hits - inval) / n, tau=float(np.mean(taus[len(taus)//4:])))
    return res


### `certa_experiments.py`

In [ ]:
%%writefile certa_experiments.py
"""
certa_experiments.py
Runs every experiment reported in the paper and writes results/*.json.
E1 main comparison, E2 ablation, E3 sensitivity (alpha, V), E4 robustness (attack rate,
Byzantine cells, sybil budget / aged accounts), E5 generalisation (PAWS shift, workload
shift), E6 efficiency, E7 statistics, E8 risk-control calibration, E9 failure analysis.
"""
import os, json, time, itertools
import numpy as np
from multiprocessing import Pool
from scipy import stats
from certa_sim import (Assets, run_one, paws_shift, METHODS, ABLATIONS, DEFAULT_CFG)

OUT = "results"
FAST = os.environ.get("CERTA_FAST", "0") == "1"      # quick smoke run: fewer seeds
SEEDS10 = [1, 2, 3] if FAST else list(range(1, 11))
SEEDS5 = [1, 2] if FAST else list(range(1, 6))
_A = None


def _init():
    global _A
    _A = Assets()


def _job(args):
    method, seed, cfg_over, flag_over = args
    t = time.perf_counter()
    r = run_one(_A, method, seed, cfg_over=cfg_over, flag_over=flag_over)
    r["wall"] = time.perf_counter() - t
    for k in ("lat_sample", "tseries", "grp_n", "grp_fr", "taus_series", "adv_cells_list"):
        if k in r and method not in ("CERTA", "Static-C"):
            r.pop(k)
    return r


def run_jobs(jobs, procs=2):
    with Pool(procs, initializer=_init) as p:
        return p.map(_job, jobs, chunksize=1)


def agg(rs, keys=("valid_hit", "frr", "asr", "poison_rate", "mean_lat", "p95_lat", "energy",
                  "llm_per_req", "cloud_share", "remote_share", "verif_per_req", "mean_tau", "p99_lat")):
    out = {}
    for k in keys:
        v = np.array([r[k] for r in rs if k in r and r[k] is not None], float)
        v = v[~np.isnan(v)] if len(v) else v
        if len(v):
            boot = np.random.default_rng(0).choice(v, size=(2000, len(v))).mean(1)
            out[k] = dict(mean=float(v.mean()), sd=float(v.std(ddof=1)) if len(v) > 1 else 0.0,
                          lo=float(np.percentile(boot, 2.5)), hi=float(np.percentile(boot, 97.5)),
                          vals=v.tolist())
    return out


def worst_group(r, min_n=200):
    n = np.array(r["grp_n"]); f = np.array(r["grp_fr"]); m = n >= min_n
    return float((f[m] / n[m]).max())


def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for i, k in enumerate(order):
        run = max(run, (m - i) * pvals[k]); adj[k] = min(1.0, run)
    return adj


def extra_experiments(procs=2):
    out = {}
    pols = [("Static-C", None), ("CERTA w/o probation", dict(probation=False)), ("CERTA", None)]
    jobs = [("Static-C" if nm == "Static-C" else "CERTA", s, None, fo) for nm, fo in pols for s in SEEDS10]
    res = run_jobs(jobs, procs)
    keys = ("adv_inserted", "adv_activated", "adv_removed_trust", "adv_served", "adv_cells_mean",
            "adv_cells_max", "adv_entries_served", "asr", "adv_occupancy", "valid_hit")
    out["E10_spread"] = {nm: agg(res[i * len(SEEDS10):(i + 1) * len(SEEDS10)], keys) for i, (nm, _) in enumerate(pols)}
    gs = [0.99, 0.995, 0.999, 0.9995]
    jobs = [("CERTA", s, dict(gamma=gm), None) for gm in gs for s in SEEDS5]
    res = run_jobs(jobs, procs)
    out["E3_gamma"] = {str(gm): agg(res[i * len(SEEDS5):(i + 1) * len(SEEDS5)]) for i, gm in enumerate(gs)}
    # imperfect answer-comparison judge
    jud = [("none", dict()), ("fa 0.10", dict(judge_fa=0.10)), ("fa 0.20", dict(judge_fa=0.20)),
           ("fd 0.10", dict(judge_fd=0.10)), ("fd 0.10, corrected", dict(judge_fd=0.10, judge_correct=True)),
           ("fd 0.20", dict(judge_fd=0.20)), ("fd 0.20, corrected", dict(judge_fd=0.20, judge_correct=True)),
           ("fa 0.10 + fd 0.10", dict(judge_fa=0.10, judge_fd=0.10)),
           ("fa 0.10 + fd 0.10, corrected", dict(judge_fa=0.10, judge_fd=0.10, judge_correct=True))]
    jobs = [("CERTA", s, co, None) for _, co in jud for s in SEEDS5]
    res = run_jobs(jobs, procs)
    out["E11_judge"] = {nm: agg(res[i * len(SEEDS5):(i + 1) * len(SEEDS5)]) for i, (nm, _) in enumerate(jud)}
    # adaptive forgery
    jobs = [("CERTA", s, dict(byz_cells=f, byz_mode="adaptive"), fo) for f in (1, 2)
            for fo in (None, dict(agg="mean")) for s in SEEDS5]
    res = run_jobs(jobs, procs)
    out["E12_adaptive_byz"] = {}
    k = 0
    for f in (1, 2):
        for nm in ("trimmed", "mean"):
            out["E12_adaptive_byz"][f"{nm} f={f}"] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    return out


def main(procs=2):
    os.makedirs(OUT, exist_ok=True)
    R = {}
    t0 = time.time()
    # ---------------- E1 main comparison ----------------
    meths = list(METHODS.keys())
    jobs = [(m, s, None, None) for m in meths for s in SEEDS10]
    res = run_jobs(jobs, procs)
    E1 = {m: [r for (mm, _, _, _), r in zip(jobs, res) if mm == m] for m in meths}
    R["E1"] = {m: agg(E1[m]) for m in meths}
    for m in ("CERTA", "Static-C"):
        R["E1"][m]["worst_group"] = [worst_group(r) for r in E1[m]]
        R["E1"][m]["lat_sample"] = E1[m][0]["lat_sample"]
        R["E1"][m]["tseries"] = E1[m][0]["tseries"]
        R["E1"][m]["taus_series"] = E1[m][0]["taus_series"]
        R["E1"][m]["grp"] = [E1[m][0]["grp_n"], E1[m][0]["grp_fr"]]
    for m in ("Exact", "Static-L", "CRC-Off", "PE-Online"):
        pass
    R["E1"]["CRC-Off"]["crc_tau"] = [r["crc_tau"] for r in E1["CRC-Off"]]
    print(f"E1 done {time.time()-t0:.0f}s", flush=True)

    # ---------------- E7 statistics (paired by seed) ----------------
    st = {}
    for metric, better in [("frr", "lower"), ("asr", "lower"), ("valid_hit", "higher"),
                           ("p95_lat", "lower"), ("energy", "lower")]:
        rows, ps = [], []
        for m in meths:
            if m == "CERTA":
                continue
            a = np.array([r[metric] for r in E1["CERTA"]]); b = np.array([r[metric] for r in E1[m]])
            d = a - b
            if np.allclose(d, 0):
                p, W = 1.0, 0.0
            else:
                W, p = stats.wilcoxon(a, b, zero_method="zsplit")
            npos = (d > 0).sum(); nneg = (d < 0).sum()
            rbc = (npos - nneg) / len(d)          # matched-pairs rank-biserial (sign form)
            ranks = stats.rankdata(np.abs(d)); rp = ranks[d > 0].sum(); rn = ranks[d < 0].sum()
            rbc = float((rp - rn) / ranks.sum()) if ranks.sum() > 0 else 0.0
            rows.append(dict(baseline=m, mean_diff=float(d.mean()), W=float(W), p=float(p), rbc=rbc))
            ps.append(p)
        adj = holm(np.array(ps))
        for r_, pa in zip(rows, adj):
            r_["p_holm"] = float(pa)
        st[metric] = rows
    R["E7"] = st

    # ---------------- E2 ablation ----------------
    ab_jobs = [("CERTA", s, None, None) for s in SEEDS10]
    for name, fo in ABLATIONS.items():
        ab_jobs += [("CERTA", s, None, fo) for s in SEEDS10]
    # robust aggregation also under one forged-report cell
    ab_jobs += [("CERTA", s, dict(byz_cells=1), None) for s in SEEDS10]
    ab_jobs += [("CERTA", s, dict(byz_cells=1), dict(agg="mean")) for s in SEEDS10]
    res = run_jobs(ab_jobs, procs)
    names = ["CERTA (full)"] + list(ABLATIONS.keys()) + ["CERTA (full), f=1", "w/o robust aggregation, f=1"]
    E2 = {}
    for i, nm in enumerate(names):
        rs = res[i * len(SEEDS10):(i + 1) * len(SEEDS10)]
        E2[nm] = agg(rs)
        if "grp_n" in rs[0]:
            E2[nm]["worst_group"] = [worst_group(r) for r in rs]
    R["E2"] = E2
    print(f"E2 done {time.time()-t0:.0f}s", flush=True)

    # ---------------- E3 + E8 alpha sweep (risk-control calibration) ----------------
    alphas = [0.01, 0.02, 0.05, 0.10]
    jobs = [(m, s, dict(alpha=a), None) for a in alphas for m in ("CERTA", "CRC-Off", "PE-Online")
            for s in SEEDS5]
    res = run_jobs(jobs, procs)
    E8 = {}
    for (m, s, co, _), r in zip(jobs, res):
        E8.setdefault(m, {}).setdefault(str(co["alpha"]), []).append(r)
    R["E8"] = {m: {a: agg(v) for a, v in d.items()} for m, d in E8.items()}
    # verification-rate, margin-gain and trust-threshold sensitivity
    sens = [("p_v", [0.02, 0.05, 0.10, 0.20]), ("W_R", [50.0, 200.0, 800.0, 3200.0]),
            ("th_act", [0.4, 0.6, 0.8, 1.0, 1.2])]
    jobs = []
    for key, vals in sens:
        for v in vals:
            co = {key: v, "th_share": v} if key == "th_act" else {key: v}
            if key == "th_act":
                co.update(sybils=8)
            jobs += [("CERTA", s, co, None) for s in SEEDS5]
    res = run_jobs(jobs, procs)
    E3 = {}; k = 0
    for key, vals in sens:
        for v in vals:
            E3.setdefault(key, {})[str(v)] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    R["E3"] = E3
    # tuned static-threshold frontier (strongest fixed-threshold competitor)
    taus = [0.80, 0.84, 0.88, 0.90, 0.92, 0.94, 0.96]
    jobs = [("Static-C", s, dict(tau_static=t), None) for t in taus for s in SEEDS5]
    res = run_jobs(jobs, procs)
    R["E1_static_sweep"] = {str(t): agg(res[i * len(SEEDS5):(i + 1) * len(SEEDS5)]) for i, t in enumerate(taus)}
    print(f"E3 done {time.time()-t0:.0f}s", flush=True)

    # ---------------- E4 robustness ----------------
    rates = [0.0, 0.02, 0.05, 0.10, 0.20]
    rm = ["Static-C", "CRC-Off", "PE-Online", "CERTA", "CERTA-noprob"]
    jobs = []
    for a in rates:
        for m in rm:
            for s in SEEDS5:
                if m == "CERTA-noprob":
                    jobs.append(("CERTA", s, dict(attack_frac=a), dict(probation=False)))
                else:
                    jobs.append((m, s, dict(attack_frac=a), None))
    res = run_jobs(jobs, procs)
    E4a = {}
    k = 0
    for a in rates:
        for m in rm:
            E4a.setdefault(m, {})[str(a)] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    R["E4_attack"] = E4a
    jobs = [("CERTA", s, dict(byz_cells=f), fo) for f in (0, 1, 2) for fo in (None, dict(agg="mean"))
            for s in SEEDS5]
    res = run_jobs(jobs, procs)
    E4b = {}; k = 0
    for f in (0, 1, 2):
        for nm in ("trimmed", "mean"):
            E4b.setdefault(nm, {})[str(f)] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    R["E4_byz"] = E4b
    syb = [2, 4, 6, 8, 12]
    trusts = [None, 0.5, 0.9]
    jobs = [("CERTA", s, dict(sybils=sb, attacker_trust=tr), None) for tr in trusts for sb in syb
            for s in SEEDS5]
    res = run_jobs(jobs, procs)
    E9 = {}; k = 0
    for tr in trusts:
        for sb in syb:
            E9.setdefault(str(tr), {})[str(sb)] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    R["E9_sybil"] = E9
    print(f"E4/E9 done {time.time()-t0:.0f}s", flush=True)

    # ---------------- E5 generalisation ----------------
    A = Assets()
    E5 = {}
    for a in (0.02, 0.05, 0.10):
        crc = np.mean(R["E8"]["CRC-Off"][str(a)]["mean_tau"]["vals"]) if False else None
        taus = [r for r in E8["CRC-Off"][str(a)]]
        crc_tau = float(np.mean([r["crc_tau"] for r in taus]))
        reps = [paws_shift(A, alpha=a, tau_crc=crc_tau, seed=s) for s in SEEDS5]
        E5[str(a)] = {m: dict(frr=float(np.mean([x[m]["frr"] for x in reps])),
                              frr_sd=float(np.std([x[m]["frr"] for x in reps], ddof=1)),
                              valid_hit=float(np.mean([x[m]["valid_hit"] for x in reps])),
                              tau=float(np.mean([x[m]["tau"] for x in reps])))
                      for m in reps[0]}
    R["E5_paws"] = E5
    shifts = [("zipf 0.7", dict(zipf_s=0.7)), ("zipf 1.1", dict(zipf_s=1.1)),
              ("4 cells", dict(n_cells=4)), ("10 cells", dict(n_cells=10, req_per_cell=4000)),
              ("cache 500", dict(cache_cap=500)), ("local mix 0.7", dict(local_mix=0.7))]
    jobs = [(m, s, co, None) for _, co in shifts for m in ("Static-C", "CRC-Off", "CERTA") for s in SEEDS5]
    res = run_jobs(jobs, procs)
    E5b = {}; k = 0
    for nm, _ in shifts:
        for m in ("Static-C", "CRC-Off", "CERTA"):
            E5b.setdefault(nm, {})[m] = agg(res[k:k + len(SEEDS5)]); k += len(SEEDS5)
    R["E5_workload"] = E5b
    # oracle-tuned static threshold (0.92 matched CERTA risk in-distribution): does it transfer?
    jobs = [("Static-C", s, dict(co, tau_static=0.92), None) for _, co in shifts for s in SEEDS5]
    res = run_jobs(jobs, procs)
    for i, (nm, _) in enumerate(shifts):
        R["E5_workload"][nm]["Static-C (0.92)"] = agg(res[i * len(SEEDS5):(i + 1) * len(SEEDS5)])
    sP, lP = A.paws_sim.astype(float), A.paws_lab.astype(bool)
    R["E5_paws_static"] = {str(t): dict(frr=float(np.mean((sP >= t) & ~lP)), valid_hit=float(np.mean((sP >= t) & lP)))
                           for t in (0.80, 0.92, 0.96, 0.98, 0.99)}
    print(f"E5 done {time.time()-t0:.0f}s", flush=True)

    # ---------------- E10 cross-cell spread of planted entries; decay sweep ----------------
    R.update(extra_experiments(procs))

    # ---------------- E6 efficiency ----------------
    E6 = {}
    for m in meths:
        w = [r["wall"] for r in E1[m]]
        n_req = DEFAULT_CFG["n_cells"] * DEFAULT_CFG["req_per_cell"] * 1.05
        E6[m] = dict(us_per_req=float(np.mean(w) / n_req * 1e6), wall=float(np.mean(w)))
    R["E6"] = E6
    R["runtime_s"] = time.time() - t0
    R["cfg"] = DEFAULT_CFG
    json.dump(R, open(os.path.join(OUT, "results.json"), "w"))
    print(f"ALL DONE {time.time()-t0:.0f}s", flush=True)
    return R


if __name__ == "__main__":
    main()


### `certa_diagrams.py`

In [ ]:
%%writefile certa_diagrams.py
"""
certa_diagrams.py  -  grayscale schematic figures (Fig. 1 architecture, Fig. 2 request pipeline).
Every arrow is drawn between explicit anchor points so that directions can be audited.
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Polygon, Circle

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
                     "font.size": 8, "mathtext.fontset": "stix"})


def box(ax, x, y, w, h, text, fc="white", lw=0.9, fs=7.5, ls="-", bold=False, ec="black"):
    p = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.25,rounding_size=0.9",
                       fc=fc, ec=ec, lw=lw, ls=ls)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", linespacing=1.15)
    return dict(l=(x, y + h / 2), r=(x + w, y + h / 2), t=(x + w / 2, y + h), b=(x + w / 2, y),
                x=x, y=y, w=w, h=h)


def arrow(ax, p, q, ls="-", lw=0.9, style="-|>", ms=7, both=False, color="black", rad=0.0):
    a = FancyArrowPatch(p, q, arrowstyle=("<|-|>" if both else style), mutation_scale=ms,
                        lw=lw, ls=ls, color=color, shrinkA=0, shrinkB=0,
                        connectionstyle=f"arc3,rad={rad}")
    ax.add_patch(a)


def poly_arrow(ax, pts, ls="-", lw=0.9, ms=7):
    for i in range(len(pts) - 2):
        ax.plot([pts[i][0], pts[i + 1][0]], [pts[i][1], pts[i + 1][1]], color="black", lw=lw, ls=ls,
                solid_capstyle="butt")
    arrow(ax, pts[-2], pts[-1], ls=ls, lw=lw, ms=ms)


def label(ax, x, y, t, fs=6.8, ha="center", style="italic", bg=True):
    ax.text(x, y, t, ha=ha, va="center", fontsize=fs, style=style,
            bbox=dict(fc="white", ec="none", pad=0.6) if bg else None)


# ----------------------------------------------------------------------------- Fig. 1
def fig_architecture(path):
    fig, ax = plt.subplots(figsize=(7.16, 3.75))
    ax.set_xlim(0, 100); ax.set_ylim(0, 56); ax.axis("off")
    # top tier
    cloud = box(ax, 4, 47, 38, 7, "Cloud LLM tier\n(large model, WAN round trip, elastic capacity)", fc="#e6e6e6")
    aggr = box(ax, 56, 47, 40, 7, "Federated risk calibration\n(trimmed aggregation  +  risk-deficit margin)",
               fc="#e6e6e6")
    # cells
    cells = []
    xs = [2, 35, 68]
    names = ["Edge cell 1", "Edge cell 2", "Edge cell C"]
    for x, nm in zip(xs, names):
        outer = FancyBboxPatch((x, 15), 30, 23, boxstyle="round,pad=0.3,rounding_size=1.2",
                               fc="#f5f5f5", ec="black", lw=1.0)
        ax.add_patch(outer)
        ax.text(x + 15, 36.3, nm, ha="center", va="center", fontsize=7.5, fontweight="bold")
        c1 = box(ax, x + 2, 26.5, 26, 7.3, "Semantic cache\n+ trust ledger (probation)", fs=7)
        c2 = box(ax, x + 2, 17.2, 26, 6.6, "Edge LLM\n(small model, 12 streams)", fs=7)
        cells.append(dict(x=x, cache=c1, llm=c2))
    ax.text(66.5, 26.5, "...", ha="center", va="center", fontsize=11)
    # users and attacker
    for x in xs:
        for k in range(4):
            ax.add_patch(Circle((x + 7 + 5.5 * k, 6.5), 1.4, fc="white", ec="black", lw=0.8))
        arrow(ax, (x + 15, 9.0), (x + 15, 14.7))
    ax.add_patch(Polygon([[36.0, 1.2], [39.4, 1.2], [37.7, 4.0]], closed=True, fc="black"))
    label(ax, 47.5, 2.6, "sybil accounts (attacker)", fs=6.5, bg=False)
    label(ax, 17, 11.7, "queries", fs=6.5)
    # cooperative lookup between neighbouring cells (bidirectional)
    arrow(ax, (32.3, 30.0), (34.7, 30.0), both=True, ms=6)
    arrow(ax, (65.3, 22.0), (67.7, 22.0), both=True, ms=6)
    # misses offloaded to the cloud (solid, upward)
    arrow(ax, (12, 38.4), (12, 46.6))
    arrow(ax, (40, 38.4), (34, 46.6))
    label(ax, 23, 43.0, "misses offloaded", fs=6.5)
    # risk histograms up to aggregator (dashed), threshold back down (dashed)
    arrow(ax, (58, 38.4), (64, 46.6), ls="--")
    arrow(ax, (76, 38.4), (76, 46.6), ls="--")
    arrow(ax, (90, 46.6), (90, 38.4), ls="--")
    label(ax, 70, 43.0, "risk histograms", fs=6.5)
    label(ax, 92.5, 43.0, r"$\hat{\tau}_t$", fs=8, style="normal")
    fig.tight_layout(pad=0.2)
    fig.savefig(path, dpi=400); plt.close(fig)


# ----------------------------------------------------------------------------- Fig. 2
def fig_pipeline(path):
    fig, ax = plt.subplots(figsize=(7.16, 4.5))
    ax.set_xlim(0, 101); ax.set_ylim(0, 66); ax.axis("off")
    W, H = 15, 6.4
    q = box(ax, 2, 52, W, H, "Query $q$\nfrom user $j$")
    e = box(ax, 22, 52, W, H, "Embedding\n$\\mathbf{e}_q$ (MiniLM)")
    c = box(ax, 42, 52, W + 2, H, "Nearest eligible entry\n$k^\\star$, similarity $s^\\star$")
    # decision diamond
    dx, dy, dw, dh = 72, 55.2, 9.5, 5.6
    ax.add_patch(Polygon([[dx - dw, dy], [dx, dy + dh], [dx + dw, dy], [dx, dy - dh]], closed=True,
                         fc="#e6e6e6", ec="black", lw=0.9))
    ax.text(dx, dy, "$s^\\star \\geq \\hat{\\tau}_t$ ?", ha="center", va="center", fontsize=7.5)
    arrow(ax, q["r"], e["l"]); arrow(ax, e["r"], c["l"]); arrow(ax, c["r"], (dx - dw, dy))
    # hit branch (right)
    hit = box(ax, 85, 38, 13, H, "Serve cached\nanswer")
    poly_arrow(ax, [(dx + dw, dy), (91.5, dy), (91.5, hit["y"] + H + 0.3)])
    label(ax, 88.0, dy + 1.8, "yes", fs=6.8)
    ver = box(ax, 85, 25, 13, H, "Shadow check\nw.p. $p$", ls="--")
    arrow(ax, hit["b"], (91.5, ver["y"] + H + 0.3), ls="--")
    # miss branch (down)
    tier = box(ax, 64.5, 38, 15, H, "Tier selection\n(edge or cloud)")
    arrow(ax, (dx, dy - dh), (dx, tier["y"] + H + 0.3))
    label(ax, dx + 2.6, 45.9, "no", fs=6.8)
    fresh = box(ax, 64.5, 25, 15, H, "Fresh LLM answer\nreturned to user")
    arrow(ax, tier["b"], (dx, fresh["y"] + H + 0.3))
    comp = box(ax, 40, 25, 18, H, "Compare with $k^\\star$ and\nprobation entry")
    arrow(ax, fresh["l"], comp["r"])
    ins = box(ax, 19, 25, 15, H, "Insert entry\n(probation state)")
    arrow(ax, comp["l"], ins["r"])
    trust = box(ax, 2, 25, 13, H, "Trust ledger\n$T_k$, $u_j$")
    arrow(ax, ins["l"], trust["r"])
    # labels into risk histogram
    hist = box(ax, 40, 11, 18, H, "Decayed IPW risk\nhistogram of cell $c$")
    arrow(ax, comp["b"], (49, hist["y"] + H + 0.3))
    label(ax, 52.6, 21.5, "label, $w=1$", fs=6.5, style="normal")
    poly_arrow(ax, [(91.5, ver["y"]), (91.5, 14.2), (hist["x"] + hist["w"] + 0.3, 14.2)], ls="--")
    label(ax, 78, 15.8, "label, $w=1/p$", fs=6.5, style="normal")
    # probation entry state change driven by the comparison
    arrow(ax, (comp["x"] + 2, comp["y"] + H), (trust["x"] + 9, trust["y"] + H + 0.2), rad=0.35, ls="-")
    label(ax, 22, 35.6, "confirm / dispute", fs=6.5)
    # aggregation and threshold feedback
    agg = box(ax, 13, 2.2, 22, H, "Trimmed aggregation over\ncells + margin $\\eta_t$")
    poly_arrow(ax, [(hist["x"], 14.2), (24, 14.2), (24, agg["y"] + H + 0.3)])
    label(ax, 31.5, 15.8, "to peers", fs=6.5)
    thr = box(ax, 85, 2.2, 13, H, "Threshold $\\hat{\\tau}_t$")
    arrow(ax, agg["r"], thr["l"], ls="--")
    poly_arrow(ax, [(thr["x"] + thr["w"] + 0.3, 5.4), (100.2, 5.4), (100.2, 64.0), (dx, 64.0),
                    (dx, dy + dh + 0.2)], ls="--")
    fig.tight_layout(pad=0.2)
    fig.savefig(path, dpi=400); plt.close(fig)


if __name__ == "__main__":
    import os
    os.makedirs("figs", exist_ok=True)
    fig_architecture("figs/fig1_architecture.png")
    fig_pipeline("figs/fig2_pipeline.png")


### `certa_figures.py`

In [ ]:
%%writefile certa_figures.py
"""
certa_figures.py - grayscale result figures generated from results/results.json.
"""
import json, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.7, "lines.linewidth": 1.1,
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5, "legend.frameon": True,
    "legend.edgecolor": "0.6", "legend.framealpha": 1.0, "savefig.dpi": 400})

STY = {  # method -> (color, marker, linestyle)
    "CERTA": ("black", "o", "-"), "Static-C": ("0.35", "s", "--"), "CRC-Off": ("0.55", "^", "-."),
    "PE-Online": ("0.2", "D", ":"), "Exact": ("0.6", "v", "-"), "Static-L": ("0.45", "<", "--"),
    "CERTA-noprob": ("0.5", "P", (0, (5, 1, 1, 1))),
}
LBL = {"CERTA-noprob": "CERTA w/o probation"}


def m(d, k):
    return d[k]["mean"], d[k]["sd"]


def legend_top(ax, ncol, y=1.02, **kw):
    return ax.legend(loc="lower center", bbox_to_anchor=(0.5, y), ncol=ncol, borderaxespad=0.3,
                     columnspacing=1.0, handlelength=2.2, **kw)


def fig_risk_control(R, path):
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.05))
    al = [0.01, 0.02, 0.05, 0.10]
    for meth in ("CERTA", "CRC-Off", "PE-Online"):
        c, mk, ls = STY[meth]
        fr = [m(R["E8"][meth][str(a)], "frr") for a in al]
        vh = [m(R["E8"][meth][str(a)], "valid_hit") for a in al]
        axs[0].errorbar(al, [x[0] for x in fr], yerr=[x[1] for x in fr], color=c, marker=mk, ls=ls, ms=3.5,
                        capsize=2, label=meth)
        axs[1].errorbar(al, [x[0] for x in vh], yerr=[x[1] for x in vh], color=c, marker=mk, ls=ls, ms=3.5,
                        capsize=2)
    axs[0].plot([0, 0.105], [0, 0.105], color="0.6", lw=0.8, ls=(0, (2, 2)), label=r"$y=\alpha$")
    axs[0].set_xlabel(r"target risk $\alpha$"); axs[0].set_ylabel("realised false-reuse rate")
    axs[1].set_xlabel(r"target risk $\alpha$"); axs[1].set_ylabel("valid hit rate")
    axs[0].set_title("(a) risk control", pad=2); axs[1].set_title("(b) utility", pad=2)
    axs[1].set_ylim(0.4, 0.9)
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0), columnspacing=0.8, handlelength=2.0)
    fig.tight_layout(rect=(0, 0, 1, 0.86), w_pad=1.2)
    fig.savefig(path); plt.close(fig)


def fig_frontier(R, path):
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.15))
    ss = R["E1_static_sweep"]
    ts = sorted(ss, key=float)
    x = [ss[t]["frr"]["mean"] for t in ts]; y = [ss[t]["valid_hit"]["mean"] for t in ts]
    a = [ss[t]["asr"]["mean"] for t in ts]
    for ax, xv in ((axs[0], x), (axs[1], a)):
        ax.plot(xv, y, color="0.4", ls="--", marker="s", ms=3, label=r"Static-C, $\tau$ swept")
    for t, xi, yi in zip(ts, x, y):
        if t in ("0.8",):
            axs[0].annotate(r"$\tau$=" + t, (xi, yi), textcoords="offset points",
                            xytext=((-8, -11) if t == "0.8" else (-3, -11)), fontsize=5.8, color="0.3")
    pts = [("CERTA", R["E1"]["CERTA"]), ("CRC-Off", R["E1"]["CRC-Off"]), ("PE-Online", R["E1"]["PE-Online"]),
           ("Exact", R["E1"]["Exact"]), ("Static-L", R["E1"]["Static-L"]),
           ("CERTA-noprob", R["E2"]["w/o probation trust"])]
    for nm, d in pts:
        c, mk, _ = STY[nm]
        for ax, k in ((axs[0], "frr"), (axs[1], "asr")):
            ax.errorbar(d[k]["mean"], d["valid_hit"]["mean"], xerr=d[k]["sd"], yerr=d["valid_hit"]["sd"],
                        color=c, marker=mk, ms=4.5 if nm == "CERTA" else 3.5, ls="none", capsize=1.5,
                        mfc=("black" if nm == "CERTA" else "white"), label=LBL.get(nm, nm))
    axs[0].axvline(0.05, color="0.6", lw=0.8, ls=":")
    axs[0].text(0.052, 0.47, r"$\alpha$", fontsize=7, color="0.3")
    axs[0].set_xlabel("false-reuse rate"); axs[0].set_ylabel("valid hit rate")
    axs[1].set_xlabel("attack success rate")
    axs[0].set_title("(a) risk-utility plane", pad=2); axs[1].set_title("(b) poisoning-utility plane", pad=2)
    for ax in axs:
        ax.set_ylim(0.45, 0.88)
    h, l = axs[1].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.995), columnspacing=0.6,
               handlelength=1.4, fontsize=6.0)
    fig.tight_layout(rect=(0, 0, 1, 0.78), w_pad=1.0)
    fig.savefig(path); plt.close(fig)


def fig_attack(R, path):
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.05))
    rates = ["0.0", "0.02", "0.05", "0.1", "0.2"]; xr = [float(r) * 100 for r in rates]
    for meth in ("Static-C", "CRC-Off", "PE-Online", "CERTA-noprob", "CERTA"):
        c, mk, ls = STY[meth]
        d = R["E4_attack"][meth]
        axs[0].errorbar(xr, [d[r]["asr"]["mean"] for r in rates], yerr=[d[r]["asr"]["sd"] for r in rates],
                        color=c, marker=mk, ls=ls, ms=3, capsize=1.5, label=LBL.get(meth, meth))
        axs[1].errorbar(xr, [d[r]["frr"]["mean"] for r in rates], yerr=[d[r]["frr"]["sd"] for r in rates],
                        color=c, marker=mk, ls=ls, ms=3, capsize=1.5)
    axs[1].axhline(0.05, color="0.6", lw=0.8, ls=":")
    for ax in axs:
        ax.set_xlabel("attack traffic (%)")
    axs[0].set_ylabel("attack success rate"); axs[1].set_ylabel("false-reuse rate")
    axs[0].set_title("(a) poisoning", pad=2); axs[1].set_title("(b) benign false reuse", pad=2)
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.0), columnspacing=0.8, handlelength=2.2)
    fig.tight_layout(rect=(0, 0, 1, 0.80), w_pad=1.2)
    fig.savefig(path); plt.close(fig)


def fig_byz_sybil(R, path):
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.45))
    fs = ["0", "1", "2"]
    for nm, c, mk, ls, lab in (("trimmed", "black", "o", "-", "trimmed (CERTA)"), ("mean", "0.5", "s", "--", "plain mean")):
        d = R["E4_byz"][nm]
        axs[0].errorbar([int(f) for f in fs], [d[f]["frr"]["mean"] for f in fs], yerr=[d[f]["frr"]["sd"] for f in fs],
                        color=c, marker=mk, ls=ls, ms=3.5, capsize=1.5, label=lab)
    axs[0].axhline(0.05, color="0.6", lw=0.8, ls=":")
    axs[0].set_xticks([0, 1, 2]); axs[0].set_xlabel("forged-report cells $f$\n(a)")
    axs[0].set_ylabel("false-reuse rate")
    axs[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=1, fontsize=6.2, handlelength=1.8)
    sy = ["2", "4", "6", "8", "12"]
    for tr, c, mk, ls, lab in (("None", "black", "o", "-", "fresh sybils"), ("0.5", "0.4", "^", "--", r"aged, $u=0.5$"),
                               ("0.9", "0.6", "s", "-.", r"aged, $u=0.9$")):
        d = R["E9_sybil"][tr]
        axs[1].errorbar([int(s_) for s_ in sy], [d[s_]["asr"]["mean"] for s_ in sy], yerr=[d[s_]["asr"]["sd"] for s_ in sy],
                        color=c, marker=mk, ls=ls, ms=3.2, capsize=1.5, label=lab)
    axs[1].set_xlabel("sybil requests / campaign\n(b)"); axs[1].set_ylabel("attack success rate")
    axs[1].set_ylim(-0.02, 0.32)
    axs[1].legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=6.0, handlelength=1.8,
                  columnspacing=0.6)
    fig.tight_layout(w_pad=1.4)
    fig.savefig(path); plt.close(fig)


def fig_latency(R, path):
    fig, ax = plt.subplots(figsize=(3.5, 2.0))
    for meth in ("Static-C", "CERTA"):
        c, _, ls = STY[meth]
        v = np.sort(np.array(R["E1"][meth]["lat_sample"]))
        ax.plot(v, np.linspace(0, 1, len(v)), color=c, ls=ls, lw=1.3, label=meth)
    ax.set_xscale("log"); ax.set_xlabel("end-to-end response time (s)"); ax.set_ylabel("empirical CDF")
    ax.axvline(0.03, color="0.7", lw=0.6, ls=":")
    ax.text(0.033, 0.06, "cache hits", fontsize=6.3, color="0.3")
    ax.text(1.6, 0.06, "LLM-served misses", fontsize=6.3, color="0.3")
    ax.legend(loc="center left", fontsize=6.5)
    fig.tight_layout()
    fig.savefig(path); plt.close(fig)


def fig_timeseries(R, path):
    fig, ax1 = plt.subplots(figsize=(3.5, 2.0))
    ts = R["E1"]["CERTA"]["tseries"]
    t = np.array([x[0] for x in ts]); n = np.array([x[1] for x in ts]); f = np.array([x[2] for x in ts])
    fr = np.where(n > 0, f / np.maximum(n, 1), np.nan)
    ts2 = R["E1"]["Static-C"]["tseries"]
    n2 = np.array([x[1] for x in ts2]); f2 = np.array([x[2] for x in ts2]); t2 = np.array([x[0] for x in ts2])
    ax1.plot(t2, np.where(n2 > 0, f2 / np.maximum(n2, 1), np.nan), color="0.55", ls="--", marker="s", ms=2.5,
             label="Static-C false reuse")
    ax1.plot(t, fr, color="black", marker="o", ms=2.5, label="CERTA false reuse")
    ax1.axhline(0.05, color="0.6", lw=0.8, ls=":")
    ax1.set_xlabel("time (s)"); ax1.set_ylabel("windowed false-reuse rate")
    ax1.set_ylim(0, 0.16)
    lo = R["cfg"]["warmup_frac"] * t.max()
    ax1.axvspan(0, lo, color="0.92", lw=0)
    ax1.text(lo * 0.5, 0.148, "cold start", ha="center", fontsize=6.3, color="0.3")
    ax2 = ax1.twinx()
    taus = R["E1"]["CERTA"]["taus_series"] if "taus_series" in R["E1"]["CERTA"] else None
    if taus:
        ax2.step(t[:len(taus)], taus[:len(t)], where="post", color="0.3", ls="-.", lw=1.0, label=r"threshold $\hat{\tau}_t$")
        ax2.set_ylabel(r"threshold $\hat{\tau}_t$"); ax2.set_ylim(0.75, 1.0); ax2.grid(False)
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, fontsize=6.0,
               handlelength=1.8, columnspacing=0.7)
    fig.tight_layout()
    fig.savefig(path); plt.close(fig)


def fig_sensitivity(R, path):
    fig, axs = plt.subplots(2, 1, figsize=(3.5, 3.7))
    th = ["0.4", "0.6", "0.8", "1.0", "1.2"]; xv = [float(t) for t in th]
    d = R["E3"]["th_act"]
    a = axs[0]; a2 = a.twinx()
    a.errorbar(xv, [d[t]["valid_hit"]["mean"] for t in th], yerr=[d[t]["valid_hit"]["sd"] for t in th], color="black",
               marker="o", ms=3, capsize=1.5, label="valid hit rate (left)")
    a2.errorbar(xv, [d[t]["asr"]["mean"] for t in th], yerr=[d[t]["asr"]["sd"] for t in th], color="0.5", ls="--",
                marker="^", ms=3, capsize=1.5, label="attack success rate (right)")
    a.set_xlabel(r"activation threshold $\theta$ (8 sybil requests per campaign)    (a)")
    a.set_ylabel("valid hit rate"); a2.set_ylabel("attack success rate"); a2.grid(False)
    h1, l1 = a.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
    a.legend(h1 + h2, l1 + l2, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=6.2, handlelength=1.8)
    pv = ["0.02", "0.05", "0.1", "0.2"]; xp = [float(p) for p in pv]
    d = R["E3"]["p_v"]
    b = axs[1]; b2 = b.twinx()
    b.errorbar(xp, [d[p]["frr"]["mean"] for p in pv], yerr=[d[p]["frr"]["sd"] for p in pv], color="black",
               marker="o", ms=3, capsize=1.5, label="false-reuse rate (left)")
    b2.errorbar(xp, [d[p]["energy"]["mean"] for p in pv], yerr=[d[p]["energy"]["sd"] for p in pv], color="0.5",
                ls="--", marker="s", ms=3, capsize=1.5, label="energy per request (right)")
    b.set_xlabel("shadow-verification probability $p$    (b)"); b.set_ylabel("false-reuse rate")
    b2.set_ylabel("energy (J)"); b.set_ylim(0, 0.05); b2.grid(False)
    h1, l1 = b.get_legend_handles_labels(); h2, l2 = b2.get_legend_handles_labels()
    b.legend(h1 + h2, l1 + l2, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=6.2, handlelength=1.8)
    fig.tight_layout(h_pad=1.0)
    fig.savefig(path); plt.close(fig)


def fig_paws(R, path):
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.1))
    names = ["Static 0.80", "Static 0.92", "Static 0.99", "CRC-Off", "CERTA"]
    hatches = ["", "////", "xxxx", "....", ""]
    fcs = ["0.85", "0.7", "white", "white", "black"]
    for j, a in enumerate(["0.05", "0.1"]):
        P = R["E5_paws"][a]; S = R["E5_paws_static"]
        fr = [S["0.8"]["frr"], S["0.92"]["frr"], S["0.99"]["frr"], P["CRC-Off"]["frr"], P["CERTA"]["frr"]]
        vh = [S["0.8"]["valid_hit"], S["0.92"]["valid_hit"], S["0.99"]["valid_hit"], P["CRC-Off"]["valid_hit"],
              P["CERTA"]["valid_hit"]]
        for i in range(5):
            axs[j].bar(i - 0.2, fr[i], 0.38, color=fcs[i], edgecolor="black", hatch=hatches[i], lw=0.6)
            axs[j].bar(i + 0.2, vh[i], 0.38, color="white", edgecolor="0.4", lw=0.6, ls="--")
        axs[j].axhline(float(a), color="0.4", lw=0.8, ls=":")
        axs[j].text(4, max(fr[4], vh[4], float(a)) + 0.03, f"{fr[4]:.3f}\n{vh[4]:.3f}", ha="center", va="bottom", fontsize=5.5)
        axs[j].set_xticks(range(5)); axs[j].set_xticklabels(names, rotation=40, ha="right", fontsize=6)
        axs[j].set_title(rf"$\alpha={a}$", pad=2); axs[j].set_ylim(0, 0.62)
    axs[0].set_ylabel("rate on PAWS pairs")
    from matplotlib.patches import Patch
    hd = [Patch(fc="0.7", ec="black", label="false reuse (solid edge)"),
          Patch(fc="white", ec="0.4", ls="--", label="valid hit (dashed edge)")]
    fig.legend(handles=hd, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.0), fontsize=6.3)
    fig.tight_layout(rect=(0, 0, 1, 0.88), w_pad=0.8)
    fig.savefig(path); plt.close(fig)


def fig_worstgroup(R, path):
    fig, ax = plt.subplots(figsize=(3.5, 1.95))
    data = [R["E1"]["Static-C"]["worst_group"], R["E2"]["w/o risk-deficit margin"]["worst_group"],
            R["E2"]["CERTA (full)"]["worst_group"], R["E2"]["+ group-conditional thresholds"]["worst_group"]]
    labs = ["Static-C", "CERTA w/o\nmargin", "CERTA", "CERTA +\ngroup thresholds"]
    bp = ax.boxplot(data, widths=0.5, patch_artist=True, medianprops=dict(color="black", lw=1.1),
                    flierprops=dict(marker="o", ms=2.5, mfc="white", mec="0.3"))
    for p, fc in zip(bp["boxes"], ["0.8", "0.6", "white", "0.9"]):
        p.set_facecolor(fc); p.set_edgecolor("black"); p.set_linewidth(0.7)
    ax.axhline(0.05, color="0.4", lw=0.8, ls=":")
    ax.set_yscale("log"); ax.set_ylabel("worst-group false reuse")
    ax.set_xticklabels(labs, fontsize=6.5)
    fig.tight_layout()
    fig.savefig(path); plt.close(fig)


def make_all(R, out="figs"):
    os.makedirs(out, exist_ok=True)
    fig_risk_control(R, f"{out}/fig3_risk_control.png")
    fig_frontier(R, f"{out}/fig4_frontier.png")
    fig_attack(R, f"{out}/fig5_attack.png")
    fig_byz_sybil(R, f"{out}/fig6_byz_sybil.png")
    fig_latency(R, f"{out}/fig7_latency.png")
    fig_timeseries(R, f"{out}/fig8_timeseries.png")
    fig_sensitivity(R, f"{out}/fig9_sensitivity.png")
    fig_paws(R, f"{out}/fig10_paws.png")
    fig_worstgroup(R, f"{out}/fig11_worstgroup.png")
    R["simstats"] = fig_similarity(f"{out}/fig12_similarity.png")




def fig_similarity(path, assets="certa_assets.npz"):
    z = np.load(assets)
    nb, ns, it = z["nb_idx"], z["nb_sim"], z["intent"]
    nbn = int(z["n_benign"])
    s_ok, s_bad = [], []
    for p in range(nbn):
        for j in range(1, nb.shape[1]):
            q = nb[p, j]
            if q < nbn and q != p:
                (s_ok if it[q] == it[p] else s_bad).append(ns[p, j]); break
    adv = z["adv_tgt_sim"]; ps, pl = z["paws_sim"], z["paws_lab"].astype(bool)
    fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.1), sharey=False)
    bins = np.linspace(0.3, 1.0, 36)
    axs[0].hist(s_bad, bins=bins, density=True, histtype="stepfilled", color="0.82", ec="0.4", lw=0.6,
                label="nearest, other intent")
    axs[0].hist(s_ok, bins=bins, density=True, histtype="step", color="black", lw=1.1, label="nearest, same intent")
    axs[0].hist(adv, bins=bins, density=True, histtype="step", color="0.35", lw=1.0, ls="--",
                label="adversarial key to target")
    axs[0].set_xlabel("cosine similarity\n(a) QQP pool"); axs[0].set_ylabel("density")
    axs[1].hist(ps[~pl], bins=bins, density=True, histtype="stepfilled", color="0.82", ec="0.4", lw=0.6,
                label="non-paraphrase")
    axs[1].hist(ps[pl], bins=bins, density=True, histtype="step", color="black", lw=1.1, label="paraphrase")
    axs[1].set_xlabel("cosine similarity\n(b) PAWS pairs")
    axs[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=5.8, handlelength=1.6)
    axs[1].legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=5.8, handlelength=1.6)
    fig.tight_layout(w_pad=0.8)
    fig.savefig(path); plt.close(fig)
    return dict(ok_med=float(np.median(s_ok)), bad_med=float(np.median(s_bad)), n_ok=len(s_ok), n_bad=len(s_bad),
                bad_above_090=float(np.mean(np.array(s_bad) >= 0.9)), ok_above_090=float(np.mean(np.array(s_ok) >= 0.9)),
                bad_above_080=float(np.mean(np.array(s_bad) >= 0.8)), ok_above_080=float(np.mean(np.array(s_ok) >= 0.8)),
                paws_neg_above_095=float(np.mean(ps[~pl] >= 0.95)), paws_pos_above_095=float(np.mean(ps[pl] >= 0.95)),
                adv_above_090=float(np.mean(adv >= 0.9)))


if __name__ == "__main__":
    R = json.load(open("results/results.json")); make_all(R); json.dump(R, open("results/results.json", "w"))


## 1. Data assets (one-off; skipped if `certa_assets.npz` already exists)

In [ ]:
import os, json, time
from certa_data import build_assets
t0 = time.time()
if not os.path.exists("certa_assets.npz"):
    build_assets()
print(json.dumps(json.load(open("certa_assets_meta.json")), indent=1))
print(f"assets ready in {time.time()-t0:.1f} s")

## 2. Experiments E1–E12

In [ ]:
import importlib, certa_experiments
importlib.reload(certa_experiments)
t0 = time.time()
certa_experiments.main(procs=max(1, os.cpu_count() or 1))
print(f"experiments finished in {(time.time()-t0)/60:.1f} min")

## 3. Figures

In [ ]:
import certa_diagrams, certa_figures
os.makedirs("figs", exist_ok=True)
certa_diagrams.fig_architecture("figs/fig1_architecture.png")
certa_diagrams.fig_pipeline("figs/fig2_pipeline.png")
R = json.load(open("results/results.json"))
certa_figures.make_all(R)
json.dump(R, open("results/results.json", "w"))
print(sorted(os.listdir("figs")))

## 4. Main comparison (mean ± sd over seeds)

In [ ]:
import pandas as pd
R = json.load(open("results/results.json"))
rows = []
for m, d in R["E1"].items():
    g = lambda k: f"{d[k]['mean']:.3f} ± {d[k]['sd']:.3f}" if isinstance(d.get(k), dict) else d.get(k)
    rows.append(dict(method=m, valid_hit=g("valid_hit"), FRR=g("frr"), ASR=g("asr"),
                     mean_latency_s=g("mean_lat"), p95_s=g("p95_lat"), energy_J=g("energy")))
pd.DataFrame(rows)

## 5. View figures

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir("figs"), key=lambda s: int(s.split("_")[0][3:])):
    print(f); display(Image(f"figs/{f}", width=620))

## 6. Download everything

In [ ]:
import shutil
import zipfile, glob
with zipfile.ZipFile("certa_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/*") + glob.glob("figs/*") + glob.glob("certa_*.py") + ["certa_assets_meta.json"]:
        z.write(f)
try:
    from google.colab import files; files.download("certa_outputs.zip")
except Exception:
    print("certa_outputs.zip written to the working directory")